# Базовый пайплайн: скользящий mPPCA → VaR → бэктестирование

Полный пайплайн оценки рыночного риска на основе **смеси вероятностных PCA (mPPCA)**,
обученной скользящим окном по логдоходностям 79 акций S&P 500 (2005–2024).

Проверяем две гипотезы: соответствует ли частота нарушений VaR заявленному уровню α
(тест Купика) и независимы ли нарушения во времени (тест Кристоффэрсена).

> Ноутбук загружает артефакты из `main.py` — переобучение не требуется.

In [ ]:
import sys, warnings
sys.path.insert(0, '..')
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns
from scipy.stats import norm, kurtosis, skew, probplot

from src.data.loader import download
from src.data.preprocessing import get_returns_array
from src.models.rolling import load as load_result
from src.models.var import compute_var_multi_level
from src.backtesting.portfolios import generate_diversified, generate_non_diversified
from src.backtesting.backtest import aggregate_results, breach_series

plt.rcParams.update({
    'figure.dpi'        : 120,
    'axes.spines.top'   : False,
    'axes.spines.right' : False,
    'axes.grid'         : True,
    'grid.alpha'        : 0.3,
    'grid.linestyle'    : '--',
})
sns.set_palette('tab10')

import os
RUN_DIR      = '../data/basic-pipeline'
ARTIFACT     = f'{RUN_DIR}/rolling_fit.npz'
WINDOW       = 350
ALPHAS       = [0.05, 0.01]
N_PORTFOLIOS = 200

os.makedirs(RUN_DIR, exist_ok=True)

## 1. Загрузка данных

In [12]:
prices = download(cache=True)
returns, dates, tickers = get_returns_array(prices)
T, D = returns.shape

eq_w         = np.ones(D) / D
port_ret_all = returns @ eq_w   # равновзвешенный портфель, вся история

print(f'Матрица доходностей : {T} дней × {D} активов')
print(f'Период              : {dates[0].date()} → {dates[-1].date()}')
print(f'Тикеры (первые 10)  : {", ".join(tickers[:10])} …')

Loading cached data from /Users/ivan/Projects/mipt/ms-mppca-coupling-diploma/notebooks/../data/sp500_adj_close.csv
Матрица доходностей : 5031 дней × 79 активов
Период              : 2005-01-04 → 2024-12-30
Тикеры (первые 10)  : AAPL, ABT, AEP, ALB, AMT, AMZN, APD, AXP, BA, BAC …


## 2. Анализ распределения логдоходностей

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

# --- Временной ряд ---
ax = axes[0]
ax.plot(dates, port_ret_all, lw=0.45, color='steelblue')
ax.set_title('Логдоходности равновзвешенного портфеля')
ax.set_xlabel('Дата')
ax.set_ylabel('Логдоходность')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(4))

# --- Эмпирическая плотность vs нормальное ---
ax = axes[1]
ax.hist(port_ret_all, bins=120, density=True, alpha=0.65, label='Эмпирическая')
x = np.linspace(port_ret_all.min(), port_ret_all.max(), 400)
ax.plot(x, norm.pdf(x, port_ret_all.mean(), port_ret_all.std()),
        'r-', lw=2, label='Нормальное')
ax.set_title('Распределение доходностей')
ax.set_xlabel('Логдоходность')
ax.set_ylabel('Плотность')
ax.set_xlim(-0.08, 0.08)
ax.legend()

# --- Q-Q график ---
ax = axes[2]
probplot(port_ret_all, dist='norm', plot=ax)
ax.set_title('Нормальный Q-Q график')
ax.set_xlabel('Теоретические квантили')
ax.set_ylabel('Выборочные квантили')

plt.tight_layout()
plt.savefig(f'{RUN_DIR}/fig_returns.png', bbox_inches='tight')
plt.show()

pd.Series({
    'Среднее'            : port_ret_all.mean(),
    'Стд. отклонение'    : port_ret_all.std(),
    'Асимметрия'         : skew(port_ret_all),
    'Избыточный эксцесс' : kurtosis(port_ret_all),
}).rename('Значение').to_frame()

## 3. Загрузка результатов скользящей подгонки mPPCA и вычисление VaR

In [14]:
result = load_result(ARTIFACT)
T_out = result.means_hist.shape[0]
K     = result.means_hist.shape[1]
q     = result.W_hist.shape[-1]

oos_dates    = dates[WINDOW : WINDOW + T_out]
oos_ret      = returns[WINDOW : WINDOW + T_out]   # (T_out, D)
port_ret_oos = oos_ret @ eq_w

portfolios_div    = generate_diversified(N_PORTFOLIOS, D)
portfolios_nondiv = generate_non_diversified(N_PORTFOLIOS, D)

# VaR вычисляется один раз и переиспользуется во всех ячейках
vars_eq     = compute_var_multi_level(result, eq_w.reshape(1, -1), ALPHAS)
vars_div    = compute_var_multi_level(result, portfolios_div,      ALPHAS)
vars_nondiv = compute_var_multi_level(result, portfolios_nondiv,   ALPHAS)

print(f'Скользящих окон : {T_out}')
print(f'Кластеров K     : {K},  латентная размерность q : {q}')
print(f'OOS период      : {oos_dates[0].date()} → {oos_dates[-1].date()}')

Скользящих окон : 4681
Кластеров K     : 2,  латентная размерность q : 3
OOS период      : 2006-05-25 → 2024-12-30


## 4. Скользящее логарифмическое правдоподобие

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
ax.plot(oos_dates, result.llh_hist, lw=0.8, color='steelblue')
ax.set_title('Скользящее логарифмическое правдоподобие mPPCA')
ax.set_xlabel('Дата')
ax.set_ylabel('Log-likelihood')
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/fig_llh.png', bbox_inches='tight')
plt.show()

## 5. Динамика весов компонент смеси (индикаторы режимов)

In [ ]:
fig, axes = plt.subplots(2, 1, figsize=(14, 7), sharex=True)

axes[0].plot(oos_dates, port_ret_oos, lw=0.45, color='steelblue')
axes[0].set_title('Доходности равновзвешенного портфеля')
axes[0].set_ylabel('Логдоходность')

for k in range(K):
    axes[1].plot(oos_dates, result.weights_hist[:, k], lw=1.0, label=f'Кластер {k+1}')
axes[1].set_title('Веса компонент смеси mPPCA $\\pi_k$ во времени')
axes[1].set_xlabel('Дата')
axes[1].set_ylabel('Вес $\\pi_k$')
axes[1].legend()
axes[1].xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
axes[1].xaxis.set_major_locator(mdates.YearLocator(2))

plt.tight_layout()
plt.savefig(f'{RUN_DIR}/fig_weights.png', bbox_inches='tight')
plt.show()

## 6. Шумовая дисперсия кластеров σ² во времени

In [ ]:
fig, ax = plt.subplots(figsize=(14, 4))
for k in range(K):
    ax.plot(oos_dates, result.sigma2_hist[:, k], lw=1.0, label=f'Кластер {k+1}')
ax.set_title('Шумовая дисперсия σ² кластеров mPPCA во времени')
ax.set_xlabel('Дата')
ax.set_ylabel('σ²')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
ax.xaxis.set_major_locator(mdates.YearLocator(2))
plt.tight_layout()
plt.savefig(f'{RUN_DIR}/fig_sigma2.png', bbox_inches='tight')
plt.show()

## 7. Временной ряд VaR с отметками нарушений

In [ ]:
fig, axes = plt.subplots(len(ALPHAS), 1, figsize=(16, 4 * len(ALPHAS)), sharex=True)

for ax, alpha in zip(axes, ALPHAS):
    var_series = vars_eq[alpha][0]
    hits = breach_series(port_ret_oos, var_series)

    ax.plot(oos_dates, port_ret_oos, lw=0.45, color='steelblue',
            label='Доходность портфеля')
    ax.plot(oos_dates, var_series, lw=1.2, color='tomato',
            label=f'VaR {int(alpha*100)}%')
    ax.scatter(
        oos_dates[hits == 1], port_ret_oos[hits == 1],
        s=8, color='red', zorder=5,
        label=f'Нарушение (n={hits.sum()}, {hits.mean()*100:.2f}%)'
    )
    ax.set_title(
        f'VaR {int(alpha*100)}% (mPPCA)  |  '
        f'частота нарушений {hits.mean()*100:.2f}%  (целевая {alpha*100:.0f}%)'
    )
    ax.set_xlabel('Дата')
    ax.set_ylabel('Логдоходность')
    ax.legend(loc='lower right', fontsize=8)
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
    ax.xaxis.set_major_locator(mdates.YearLocator(2))

plt.tight_layout()
plt.savefig(f'{RUN_DIR}/fig_var.png', bbox_inches='tight')
plt.show()

## 8. Распределение частоты нарушений по портфелям

In [ ]:
fig, axes = plt.subplots(1, len(ALPHAS), figsize=(14, 4))

for ax, alpha in zip(axes, ALPHAS):
    for label, portfolios, vars_m, color in [
        ('Диверсифицированные', portfolios_div,    vars_div[alpha],    'steelblue'),
        ('Концентрированные',   portfolios_nondiv, vars_nondiv[alpha], 'tomato'),
    ]:
        rates = np.array([
            breach_series(oos_ret @ portfolios[j], vars_m[j]).mean()
            for j in range(len(portfolios))
        ]) * 100
        ax.hist(rates, bins=30, alpha=0.6, label=label, color=color, density=True)

    ax.axvline(alpha * 100, color='black', ls='--', lw=1.5,
               label=f'Целевой уровень {alpha*100:.0f}%')
    ax.set_title(f'Частота нарушений — VaR {int(alpha*100)}%')
    ax.set_xlabel('Частота нарушений (%)')
    ax.set_ylabel('Плотность')
    ax.legend()

plt.tight_layout()
plt.savefig(f'{RUN_DIR}/fig_breach_dist.png', bbox_inches='tight')
plt.show()

## 9. Сводные таблицы бэктестирования

In [20]:
col_rename = {
    'breach_rate': 'Breach rate',
    'pvalue'     : 'p-value (Kupiec)',
    'pvalue_pass': 'Kupiec passed',
    'ci_95_pass' : 'CI 95% passed',
    'ci_99_pass' : 'CI 99% passed',
    'ind_pass'   : 'Christoffersen passed',
}

for label, portfolios, vars_m in [
    ('Diversified',   portfolios_div,    vars_div),
    ('Concentrated',  portfolios_nondiv, vars_nondiv),
]:
    rows = [
        aggregate_results(portfolios, oos_ret, vars_m[alpha], alpha)
        for alpha in ALPHAS
    ]
    df = pd.concat(rows)[list(col_rename)].rename(columns=col_rename).round(4)
    print(f'\n── {label} ─────────────────────────────────')
    display(df)


── Diversified ─────────────────────────────────


,Breach rate,p-value (Kupiec),Kupiec passed,CI 95% passed,CI 99% passed,Christoffersen passed
alpha=0.05,0.0592,0.005,0.0,0.0,0.045,0.0
alpha=0.01,0.0240,0.000,0.0,0.0,0.000,0.0



── Concentrated ─────────────────────────────────


,Breach rate,p-value (Kupiec),Kupiec passed,CI 95% passed,CI 99% passed,Christoffersen passed
alpha=0.05,0.0548,0.2462,0.635,0.655,0.795,0.050
alpha=0.01,0.0189,0.0004,0.005,0.005,0.005,0.235


## 10. Выводы

### Результаты

- **Концентрированные портфели, α=5%**: Купик пройден (p-value ≈ 0.25, breach rate ≈ 5.5%). Смесь гауссиан хорошо захватывает тяжёлые хвосты там, где нормальное распределение ломается.
- **Динамика весов**: компоненты смеси меняются во времени — кризис 2008 и COVID-2020 видны как режимы с резко повышенной σ². Это говорит об осмысленной идентификации режимов.
- **Скорость**: 4681 скользящее окно (79 активов) — ~15 секунд благодаря тёплому старту.

### Ограничения

- **Диверсифицированные портфели**: breach rate 5.9% / 2.4% при целевых 5% / 1% — модель недооценивает хвостовой риск. Причина: диверсифицированный портфель близок к нормальному (ЦПТ), и смесь не даёт преимущества перед простым гауссовым VaR.
- **α=1%**: Купик провален для обоих типов портфелей. Гауссовы хвосты слишком лёгкие.
- **Кристоффэрсен**: провален везде — нарушения кластеризуются. i.i.d. смесь не передаёт состояние между окнами.

### Дальнейшие шаги

Кластеризация нарушений — мотивация добавить **Markov Switching**: если переходы между
кластерами управляются марковской цепью, нарушения должны стать более независимыми.
Для α=1% стоит заменить гауссовые квантили на $t_\nu$ (см. `student-t-comparison.ipynb`).